# ISAC Resource Allocation Demo

This notebook demonstrates the ISAC resource allocation framework from:
**"Sensing as a Service in 6G Perceptive Networks: A Unified Framework for ISAC Resource Allocation"**

Authors: Fuwang Dong, Fan Liu, Yuanhao Cui, Wei Wang, Kaifeng Han, Zhiqin Wang
IEEE Transactions on Wireless Communications, 2022

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import numpy as np
import matplotlib.pyplot as plt
from src.system_model import ISACSystem
from src.detection_qos import DetectionQoS
from src.localization_qos import LocalizationQoS
from src.tracking_qos import TrackingQoS
from src.ao_solver import AOSolver
from src.fairness import FairnessMetrics

## 1. System Setup

In [ ]:
# Create ISAC system with default parameters
rng = np.random.default_rng(42)
system = ISACSystem(Nt=32, Nr=32, Q=3, K=3, L=1, fc=30e9,
                    P_total=40.0, B_total=100e6, rng=rng)

print(f"System Configuration:")
print(f"  Transmit antennas: {system.params.Nt}")
print(f"  Receive antennas: {system.params.Nr}")
print(f"  Sensing targets: {system.params.Q}")
print(f"  Communication users: {system.params.K}")
print(f"  ISAC users: {system.params.L}")
print(f"  Total objects: {system.params.M}")
print(f"  Carrier frequency: {system.params.fc/1e9:.1f} GHz")
print(f"  Total power: {system.params.P_total} W")
print(f"  Total bandwidth: {system.params.B_total/1e6:.0f} MHz")
print(f"\nNoise power: {system.N0:.2e} W/Hz")

In [ ]:
# Display target and user positions
print("Sensing Target Positions (range, angle):")
for q in range(system.params.Q):
    print(f"  Target {q}: d={system.target_positions[q]:.1f}m, θ={np.degrees(system.target_angles[q]):.1f}°")

print("\nCommunication User Positions (range, angle):")
for k in range(system.params.K):
    print(f"  User {k}: d={system.user_positions[k]:.1f}m, θ={np.degrees(system.user_angles[k]):.1f}°")

print(f"\nISAC User: d={system.isac_position:.1f}m, θ={np.degrees(system.isac_angle):.1f}°")

## 2. Detection QoS Analysis

In [ ]:
# Test detection probability vs power
detection = DetectionQoS(system, Pfa=0.01)

power_range = np.linspace(1, 30, 20)
b_fixed = np.array([30e6, 30e6, 25e6])

P_D_results = []
for p_val in power_range:
    p = np.array([p_val, p_val, p_val])
    P_D = detection.compute_detection_probability(p, b_fixed)
    P_D_results.append(P_D)

P_D_results = np.array(P_D_results)

plt.figure(figsize=(10, 6))
for q in range(system.params.Q):
    plt.plot(power_range, P_D_results[:, q], 'o-', label=f'Target {q}')

plt.xlabel('Transmit Power per Target (W)')
plt.ylabel('Detection Probability')
plt.title('Detection Probability vs Transmit Power')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.show()

## 3. Localization QoS Analysis

In [ ]:
# Test CRB vs power and bandwidth
localization = LocalizationQoS(system)

# CRB vs Power
p_values = np.linspace(1, 30, 15)
crb_range_power = []
crb_angle_power = []

for p_val in p_values:
    p = np.array([p_val, p_val, p_val])
    crb_d = localization.compute_crb_range(p, b_fixed)
    crb_theta = localization.compute_crb_angle(p, b_fixed)
    crb_range_power.append(np.mean(crb_d))
    crb_angle_power.append(np.mean(crb_theta))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.semilogy(p_values, crb_range_power, 'o-')
ax1.set_xlabel('Transmit Power (W)')
ax1.set_ylabel('Mean CRB for Range (m²)')
ax1.set_title('Range CRB vs Power')
ax1.grid(True, alpha=0.3)

ax2.semilogy(p_values, crb_angle_power, 'o-')
ax2.set_xlabel('Transmit Power (W)')
ax2.set_ylabel('Mean CRB for Angle (rad²)')
ax2.set_title('Angle CRB vs Power')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Alternating Optimization Solver

In [ ]:
# Solve for Detection QoS with max-min fairness
solver = AOSolver(system, qos_type='detection', fairness='maxmin', max_iter=20)

result = solver.solve(Gamma_c=1.0)  # 1 bps/Hz rate threshold

print("Optimization Results:")
print(f"  Converged: {result.converged}")
print(f"  Iterations: {result.iterations}")
print(f"  Objective value: {result.objective:.4f}")
print(f"\nPower Allocation (W):")
print(f"  Sensing: {result.p[:system.params.Q]}")
print(f"  Comm: {result.p[system.params.Q:system.params.Q+system.params.K]}")
print(f"  ISAC: {result.p[system.params.Q+system.params.K:]}")
print(f"  Total: {np.sum(result.p):.2f} W (budget: {system.params.P_total} W)")
print(f"\nBandwidth Allocation (MHz):")
print(f"  Sensing: {result.b[:system.params.Q]/1e6}")
print(f"  Comm: {result.b[system.params.Q:system.params.Q+system.params.K]/1e6}")
print(f"  ISAC: {result.b[system.params.Q+system.params.K:]/1e6}")
print(f"  Total: {np.sum(result.b)/1e6:.2f} MHz (budget: {system.params.B_total/1e6} MHz)")

In [ ]:
# Display QoS metrics
if result.detection_probs is not None:
    print("Detection Probabilities:")
    for q, pd in enumerate(result.detection_probs):
        print(f"  Target {q}: P_D = {pd:.4f}")

if result.comm_rates is not None:
    print("\nCommunication Rates (bps/Hz):")
    for k, rate in enumerate(result.comm_rates):
        print(f"  User {k}: R = {rate:.4f} (threshold: 1.0)")

## 5. Comparison of Different QoS Types

In [ ]:
# Solve for all three QoS types
results = solver.solve_multiple_qos(Gamma_c=1.0)

print("Comparison of Different QoS Types:")
print("=" * 50)

for qos_type, res in results.items():
    print(f"\n{qos_type.upper()} QoS:")
    print(f"  Converged: {res.converged}, Iterations: {res.iterations}")
    print(f"  Objective: {res.objective:.4f}")
    print(f"  Power allocation (sensing): {res.p[:system.params.Q]}")
    print(f"  Bandwidth allocation (sensing): {res.b[:system.params.Q]/1e6} MHz")

## 6. Tracking Simulation

In [ ]:
# Simulate tracking over time
tracking = TrackingQoS(system, dt=0.1)

# Use equal allocation for demonstration
p_equal = np.ones(system.params.Q) * 10.0
b_equal = np.ones(system.params.Q) * 30e6

pcrb_history, trace_history = tracking.simulate_tracking(p_equal, b_equal, num_steps=20)

plt.figure(figsize=(10, 6))
plt.plot(trace_history, 'o-')
plt.xlabel('Time Step')
plt.ylabel('Sum of PCRB Trace')
plt.title('Tracking Performance Over Time')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

## 7. Fairness Analysis

In [ ]:
# Analyze fairness metrics
fairness = FairnessMetrics()

if result.detection_probs is not None:
    fairness_metrics = fairness.evaluate_fairness_metrics(result.detection_probs)
    
    print("Fairness Metrics for Detection Probabilities:")
    for metric, value in fairness_metrics.items():
        print(f"  {metric}: {value:.4f}")